# Variational Linear Attention (VLA)
### From Theory to GPU-Fused Triton Kernel

---

> **Runtime**: GPU required. In Colab → `Runtime → Change runtime type → T4 GPU`

This notebook implements VLA end-to-end with **three progressively faster backends**:

| Backend | How | Expected speedup |
|---|---|---|
| Sequential | Python for-loop, one kernel launch per token | 1× (baseline) |
| Parallel Scan | Blelloch tree scan, O(log T) depth | ~2× |
| **Triton Fused** | Entire A_t loop in one CUDA kernel | **~4–5×** |

---
## 1. The Problem with Standard Linear Attention

Standard linear attention approximates the softmax kernel with a feature map φ:

$$o_t = \frac{S_t\, q_t}{z_t}, \quad S_t = S_{t-1} + \varphi(v_t)\,\varphi(k_t)^\top, \quad z_t = z_{t-1} + \varphi(k_t)$$

**The problem**: every new key overwrites equally. Memory saturates — old facts get diluted away regardless of relevance. At T=4096 the system treats step 1 and step 4095 identically.

---
## 2. The VLA Fix: Adaptive Forgetting via Sherman-Morrison

VLA replaces the fixed accumulation with a **residual-error update**:

$$S_t = S_{t-1} + \underbrace{(v_t - S_{t-1}\,k_t)}_{\text{prediction error}\; e_t}\; \alpha_t^\top$$

where $\alpha_t = A_t\,k_t$ is a *learned attention vector* controlled by an adaptive inverse matrix $A_t$.

### 2.1 What is $A_t$?

$A_t$ is the inverse of a **penalty matrix** $M_t = \lambda_t I + u_t u_t^\top$ that up-weights directions the model wants to remember.

The inverse is updated via **Sherman-Morrison** — O(d²) per step instead of O(d³) for full inversion:

$$A_t = A_{t-1} - \frac{(A_{t-1}\,u_t)(A_{t-1}\,u_t)^\top}{1 + u_t^\top A_{t-1}\,u_t}$$

This is exact, not approximate.

---
## 3. The Parallelization Challenge

**$S_t$ update** is a linear recurrence — parallelizable:

$$S_t = F_t\,S_{t-1} + G_t, \quad F_t = I - k_t\,\alpha_t^\top, \quad G_t = v_t\,\alpha_t^\top$$

Since $(F, G)$ is associative under $(F_r, G_r) \circ (F_l, G_l) = (F_r F_l,\; F_r G_l + G_r)$, we can run a **Blelloch parallel prefix scan** — O(log T) depth on GPU.

**$A_t$ update** has a data-dependent denominator — no clean parallel form. The Triton kernel solves this by *fusing the entire sequential loop into one kernel*, eliminating T separate kernel launches.

---
## Setup

In [1]:
# Triton is bundled with PyTorch >= 2.0 on CUDA — nothing extra needed
# If running locally without CUDA: pip install torch

import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"Triton  : {HAS_TRITON}")

if DEVICE == "cpu":
    print("\n⚠️  No GPU detected. Triton skipped; sequential and parallel scan will run on CPU (parallel will be SLOWER on CPU — that's expected).")

PyTorch : 2.10.0+cu128
Device  : cuda
Triton  : True


---
## Block 1 — Triton Kernel: Fused Sherman-Morrison Scan

**Key insight**: the standard A_t loop launches T separate CUDA kernels (one per token). For T=4096 that's 4096 round-trips through the CUDA scheduler. The Triton kernel runs the *entire sequence* inside one kernel, keeping A_t alive in registers.

```
One Triton program = one batch element
A_t (D×D) lives in registers throughout the T-step loop
Valid for D ≤ 128  (D² × 4B = 64 KB ≤ L1 cache)
```

Matrix-vector product `A @ u` is computed via broadcast + reduce:
```python
z = tl.sum(A * u[None, :], axis=1)   # shape (D,) — no tl.dot needed
```
Outer product `z @ z^T` via:
```python
outer = z[:, None] * z[None, :]       # shape (D, D)
```

In [2]:
# ─── Triton kernel ─────────────────────────────────────────────────────────

if HAS_TRITON:
    @triton.jit
    def _sm_scan_kernel(
        U_ptr, K_ptr, Out_ptr,
        T,
        stride_Ub, stride_Ut,
        stride_Kb, stride_Kt,
        stride_Ob, stride_Ot,
        inv_lambda0: tl.constexpr,
        inv_sqrt_D:  tl.constexpr,
        stab_eps:    tl.constexpr,
        period:      tl.constexpr,
        per_eps:     tl.constexpr,
        D:           tl.constexpr,   # power-of-2, ≤ 128
    ):
        pid  = tl.program_id(0)      # one program per batch element
        ids  = tl.arange(0, D)       # [0 .. D-1]

        # Initialise A = (1/λ₀) · I  in registers  (D × D floats)
        A = tl.where(
            ids[:, None] == ids[None, :],
            inv_lambda0, 0.0,
        ).to(tl.float32)

        # Pre-build stability / periodic nudge matrices
        stab_mat = tl.where(ids[:, None] == ids[None, :], stab_eps, 0.0).to(tl.float32)
        per_mat  = tl.where(ids[:, None] == ids[None, :], per_eps,  0.0).to(tl.float32)

        U_base = U_ptr   + pid * stride_Ub
        K_base = K_ptr   + pid * stride_Kb
        O_base = Out_ptr + pid * stride_Ob

        for t in tl.range(T):
            # Load u_t, k_t — shape (D,)
            u = tl.load(U_base + t * stride_Ut + ids).to(tl.float32) * inv_sqrt_D
            k = tl.load(K_base + t * stride_Kt + ids).to(tl.float32)

            # z = A @ u
            z = tl.sum(A * u[None, :], axis=1)        # (D,)

            # Sherman-Morrison denominator  δ = 1 + u·z
            delta = tl.sum(u * z) + 1.0
            delta = tl.maximum(delta, stab_eps)        # numerical floor

            # A ← A − (z zᵀ) / δ
            outer = z[:, None] * z[None, :]            # (D, D)
            A_new = A - outer / delta

            # Stability fallback: if δ was clamped, add ε·I instead
            A = tl.where(delta <= stab_eps, A + stab_mat, A_new)

            # Periodic nudge every `period` steps
            if (t + 1) % period == 0:
                A = A + per_mat

            # α_t = A @ k_t  — what we actually need downstream
            alpha = tl.sum(A * k[None, :], axis=1)    # (D,)
            tl.store(O_base + t * stride_Ot + ids, alpha)


    def sm_scan_triton(
        U: torch.Tensor, K: torch.Tensor,
        lambda_0: float = 1.0,
        stab_eps: float = 1e-6,
        per_eps:  float = 1e-5,
        period:   int   = 50,
    ) -> torch.Tensor:
        """
        Fused Sherman-Morrison scan.
        Args:
            U: (B, T, D)  penalty vectors
            K: (B, T, D)  key vectors
        Returns:
            alpha: (B, T, D)  alpha_t = A_t k_t for all t
        """
        B, T, D = U.shape
        assert D & (D - 1) == 0 and D <= 128, f"D must be power-of-2 ≤ 128, got {D}"
        assert U.is_contiguous() and K.is_contiguous()

        alpha = torch.empty_like(U)
        _sm_scan_kernel[(B,)](
            U, K, alpha, T,
            U.stride(0), U.stride(1),
            K.stride(0), K.stride(1),
            alpha.stride(0), alpha.stride(1),
            inv_lambda0 = 1.0 / lambda_0,
            inv_sqrt_D  = 1.0 / math.sqrt(D),
            stab_eps    = stab_eps,
            period      = period,
            per_eps     = per_eps,
            D           = D,
        )
        return alpha

    print("✅ Triton kernel compiled")
else:
    print("⚠️  Triton not available — VLATriton will be skipped in benchmarks")

✅ Triton kernel compiled


---
## Block 2 — Penalty Builder

Builds the penalty direction $u_t$ for each token. The model *learns* which directions to protect in memory by predicting $\lambda_t$ (scalar) and $u_t$ (vector) from the key:

$$M_t = \lambda_t I + u_t u_t^\top \quad \Rightarrow \quad A_t = M_t^{-1}$$

In [3]:
class PenaltyBuilder(nn.Module):
    """
    Given key vectors k ∈ ℝ^d, predicts:
      - λ_t  (scalar forgetting rate) via small MLP + softplus
      - u_t  (penalty direction)      via linear projection
    """
    def __init__(self, d_model, fixed_lambda=None, lambda_min=1e-4):
        super().__init__()
        self.fixed_lambda = fixed_lambda
        self.lambda_min   = lambda_min
        self.lambda_net   = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(),
            nn.Linear(4 * d_model, 1),
        )
        self.u_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, k):
        if self.fixed_lambda is not None:
            lam = torch.full(k.shape[:-1] + (1,), self.fixed_lambda,
                             device=k.device, dtype=k.dtype)
        else:
            lam = F.softplus(self.lambda_net(k)).clamp(min=self.lambda_min)
        u = self.u_proj(k)
        return lam, u

print("✅ PenaltyBuilder defined")

✅ PenaltyBuilder defined


---
## Block 3 — A_t Sequence (Sequential, compile-friendly)

The Sherman-Morrison loop — no `.item()` calls, so `torch.compile` can fuse it into a single CUDA stream. Uses `torch.where` for data-dependent branches instead of Python `if`.

In [4]:
def _compute_A_sequence(U_seq, lambda_0, d, stab_eps=1e-6, per_eps=1e-5, period=50):
    """
    Sequential Sherman-Morrison: A_t for t = 1..T.
    No .item() — fully torch.compile-able.

    Args:
        U_seq : (B, T, d)  penalty vectors
    Returns:
        A_all : (T, B, d, d)
    """
    B, T, _ = U_seq.shape
    device, dtype = U_seq.device, U_seq.dtype

    I   = torch.eye(d, device=device, dtype=dtype).unsqueeze(0).expand(B, -1, -1)
    A_t = (1.0 / lambda_0) * I.clone()
    A_all = []

    for t in range(T):
        u    = U_seq[:, t, :] / math.sqrt(d)          # (B, d)  — normalise
        uv   = u.unsqueeze(-1)                          # (B, d, 1)
        z    = torch.bmm(A_t, uv)                       # (B, d, 1)
        dot  = torch.bmm(uv.transpose(1, 2), z).squeeze(-1).squeeze(-1)  # (B,)
        delt = torch.clamp(1.0 + dot, min=stab_eps)    # (B,)

        upd  = torch.bmm(z, z.transpose(1, 2)) / delt.view(B, 1, 1)  # (B,d,d)
        bad  = (torch.abs(delt) < stab_eps).view(B, 1, 1)
        A_t  = torch.where(bad, A_t + stab_eps * I, A_t - upd)

        if (t + 1) % period == 0:
            A_t = A_t + per_eps * I

        A_all.append(A_t)

    return torch.stack(A_all, dim=0)   # (T, B, d, d)

print("✅ _compute_A_sequence defined")

✅ _compute_A_sequence defined


---
## Block 4 — Parallel Prefix Scan for S_t

Blelloch work-efficient scan — O(T) work, O(log T) parallel depth.

```
Up-sweep   (reduce):   compute partial sums bottom-up in a binary tree
Down-sweep (broadcast): push exclusive prefix sums top-down
```

**Why the normalization?**  
Without it, $F_t = I - k_t \alpha_t^\top$ can have eigenvalues > 1. Composing $\log_2 T$ such matrices in the tree → exponential blowup → NaN gradients at T=4096.  
Normalizing $k_t, \alpha_t$ to unit vectors makes $F_t$ a valid contraction (eigenvalue 0 in the $k$ direction, 1 elsewhere).

In [5]:
def _combine(F_l, G_l, F_r, G_r):
    """Associative operator for the (F, G) recurrence."""
    return torch.matmul(F_r, F_l), torch.matmul(F_r, G_l) + G_r


def parallel_scan(Fs, Gs):
    """
    Inclusive parallel prefix scan for S_t = F_t @ S_{t-1} + G_t,  S_0 = 0.

    Args:
        Fs, Gs : (T, B, d, d)
    Returns:
        S_all  : (T, B, d, d)  where S_all[t] = S_{t+1}
    """
    T, B, d, _ = Fs.shape

    # Pad to next power of 2
    T_pad = 1 << (T - 1).bit_length()
    pad   = T_pad - T
    if pad > 0:
        I_p = torch.eye(d, device=Fs.device, dtype=Fs.dtype) \
                   .unsqueeze(0).unsqueeze(0).expand(pad, B, -1, -1)
        Z_p = torch.zeros(pad, B, d, d, device=Fs.device, dtype=Fs.dtype)
        Fs  = torch.cat([Fs, I_p], dim=0)
        Gs  = torch.cat([Gs, Z_p], dim=0)

    Fc, Gc = Fs.clone(), Gs.clone()

    # ── Up-sweep ──────────────────────────────────────────────────────────
    stride = 1
    while stride < T_pad:
        r = torch.arange(2 * stride - 1, T_pad, 2 * stride, device=Fs.device)
        l = r - stride
        if r.numel() > 0:
            Fc[r], Gc[r] = _combine(Fc[l], Gc[l], Fc[r], Gc[r])
        stride *= 2

    # ── Down-sweep ────────────────────────────────────────────────────────
    Fc[T_pad - 1] = torch.eye(d, device=Fs.device, dtype=Fs.dtype) \
                         .unsqueeze(0).expand(B, -1, -1)
    Gc[T_pad - 1] = torch.zeros(B, d, d, device=Fs.device, dtype=Fs.dtype)

    stride = T_pad // 2
    while stride >= 1:
        r = torch.arange(2 * stride - 1, T_pad, 2 * stride, device=Fs.device)
        l = r - stride
        if r.numel() > 0:
            Fr, Gr = Fc[r].clone(), Gc[r].clone()
            Fl, Gl = Fc[l].clone(), Gc[l].clone()
            Fc[l], Gc[l] = Fr, Gr
            Fc[r], Gc[r] = _combine(Fl, Gl, Fr, Gr)
        stride //= 2

    # Inclusive: apply original (Fs, Gs) to exclusive prefix
    S_all = torch.matmul(Fs[:T], Gc[:T]) + Gs[:T]
    return S_all

print("✅ parallel_scan defined")

✅ parallel_scan defined


---
## Block 5 — Three VLA Layer Variants

All three share the same projections and output path. They differ only in how they compute α_t.

In [6]:
# ─── Shared base ───────────────────────────────────────────────────────────

class _VLABase(nn.Module):
    """Shared projections + S_t scan logic."""

    def __init__(self, d_model, lambda_0=1.0, fixed_lambda=None,
                 stab_eps=1e-6, per_eps=1e-5, period=50):
        super().__init__()
        self.d_model  = d_model
        self.lambda_0 = lambda_0
        self.stab_eps = stab_eps
        self.per_eps  = per_eps
        self.period   = period

        self.W_q      = nn.Linear(d_model, d_model)
        self.W_k      = nn.Linear(d_model, d_model)
        self.W_v      = nn.Linear(d_model, d_model)
        self.out_norm = nn.LayerNorm(d_model)
        self.W_o      = nn.Linear(d_model, d_model)
        self.penalty  = PenaltyBuilder(d_model, fixed_lambda=fixed_lambda)

    def _build_S(self, alpha_all, K, V, d, B, T, device, dtype):
        """Parallel scan for S_t given pre-computed alpha_all (T,B,d)."""
        K_tr  = K.permute(1, 0, 2)   # (T, B, d)
        V_tr  = V.permute(1, 0, 2)
        I_exp = torch.eye(d, device=device, dtype=torch.float32) \
                     .unsqueeze(0).unsqueeze(0).expand(T, B, -1, -1)

        # Normalise k and α before rank-1 products to keep F_t a contraction.
        # Without this, composing F matrices in the binary tree causes
        # exponential blowup → NaN gradients at long T.
        k_n = F.normalize(K_tr, dim=-1)
        a_n = F.normalize(alpha_all, dim=-1)

        k_alpha = torch.matmul(k_n.unsqueeze(-1), a_n.unsqueeze(-2))   # (T,B,d,d)
        v_alpha = torch.matmul(V_tr.unsqueeze(-1), a_n.unsqueeze(-2))  # (T,B,d,d)

        # Clip G to prevent S explosion
        vn = torch.norm(v_alpha, dim=(-2, -1), keepdim=True)
        v_alpha = torch.where(vn > 10.0, v_alpha * 10.0 / (vn + 1e-6), v_alpha)

        Fs = I_exp - k_alpha
        Gs = v_alpha
        return parallel_scan(Fs, Gs)   # (T, B, d, d)

    def _output(self, S_all, Q, d, B, T, dtype):
        Q_tr = Q.permute(1, 0, 2).unsqueeze(-1)             # (T, B, d, 1)
        O    = torch.matmul(S_all, Q_tr).squeeze(-1) \
                    .permute(1, 0, 2).to(dtype)              # (B, T, d)
        return self.W_o(self.out_norm(O))

print("✅ _VLABase defined")

✅ _VLABase defined


In [7]:
# ─── Variant 1: Sequential ─────────────────────────────────────────────────

class VLASequential(_VLABase):
    """Baseline: Python for-loop over T (one CUDA kernel launch per token)."""

    def forward(self, x):
        B, T, _ = x.shape
        d = self.d_model
        device, dtype = x.device, x.dtype

        Q = self.W_q(x).float()
        K = self.W_k(x).float()
        V = self.W_v(x).float()
        _, U_seq = self.penalty(K)

        I   = torch.eye(d, device=device, dtype=torch.float32).unsqueeze(0).expand(B, -1, -1)
        A_t = (1.0 / self.lambda_0) * I.clone()
        S_t = torch.zeros(B, d, d, device=device, dtype=torch.float32)
        outputs = []

        for t in range(T):
            u    = U_seq[:, t, :].float() / math.sqrt(d)
            uv   = u.unsqueeze(-1)
            z    = torch.bmm(A_t, uv)
            dot  = torch.bmm(uv.transpose(1, 2), z).squeeze(-1).squeeze(-1)
            delt = torch.clamp(1.0 + dot, min=self.stab_eps)
            upd  = torch.bmm(z, z.transpose(1, 2)) / delt.view(B, 1, 1)
            bad  = (torch.abs(delt) < self.stab_eps).view(B, 1, 1)
            A_t  = torch.where(bad, A_t + self.stab_eps * I, A_t - upd)
            if (t + 1) % self.period == 0:
                A_t = A_t + self.per_eps * I

            k_t     = K[:, t, :].float()
            v_t     = V[:, t, :].float()
            q_t     = Q[:, t, :].float()
            alpha_t = torch.bmm(A_t, k_t.unsqueeze(-1)).squeeze(-1)

            v_hat = torch.bmm(S_t, k_t.unsqueeze(-1)).squeeze(-1)
            e_t   = v_t - v_hat
            en    = torch.norm(e_t, dim=-1, keepdim=True)
            e_t   = torch.where(en > 10.0, e_t * 10.0 / (en + 1e-6), e_t)

            S_t = S_t + torch.matmul(e_t.unsqueeze(2), alpha_t.unsqueeze(1))
            outputs.append(torch.matmul(S_t, q_t.unsqueeze(2)).squeeze(2))

        O = torch.stack(outputs, dim=1).to(dtype)
        return self.W_o(self.out_norm(O))

print("✅ VLASequential defined")

✅ VLASequential defined


In [8]:
# ─── Variant 2: Parallel Scan ──────────────────────────────────────────────

class VLAParallel(_VLABase):
    """Python A_t loop + parallel prefix scan for S_t.  ~2× speedup on GPU."""

    def forward(self, x):
        B, T, _ = x.shape
        d = self.d_model
        device, dtype = x.device, x.dtype

        Q = self.W_q(x).float()
        K = self.W_k(x).float()
        V = self.W_v(x).float()
        _, U_seq = self.penalty(K)

        A_all     = _compute_A_sequence(U_seq.float(), self.lambda_0, d,
                                        self.stab_eps, self.per_eps, self.period)
        K_t       = K.permute(1, 0, 2).unsqueeze(-1)
        alpha_all = torch.matmul(A_all, K_t).squeeze(-1)   # (T, B, d)

        S_all = self._build_S(alpha_all, K, V, d, B, T, device, dtype)
        return self._output(S_all, Q, d, B, T, dtype)

print("✅ VLAParallel defined")

✅ VLAParallel defined


In [9]:
# ─── Variant 3: Triton Fused ───────────────────────────────────────────────

class VLATriton(_VLABase):
    """
    Triton kernel fuses entire A_t loop into ONE kernel launch.
    Requires CUDA + D power-of-2 ≤ 128.
    ~4-5× over sequential, ~2× over parallel scan.
    """

    def forward(self, x):
        assert HAS_TRITON and x.device.type == "cuda", \
            "VLATriton requires CUDA + Triton"
        B, T, _ = x.shape
        d = self.d_model
        device, dtype = x.device, x.dtype

        Q = self.W_q(x).float()
        K = self.W_k(x).float()
        V = self.W_v(x).float()
        _, U_seq = self.penalty(K)

        # ── Triton fused A_t → alpha_t  (one kernel launch for all T steps) ──
        U_c = U_seq.float().contiguous()
        K_c = K.float().contiguous()
        alpha_all = sm_scan_triton(
            U_c, K_c,
            lambda_0 = self.lambda_0,
            stab_eps = self.stab_eps,
            per_eps  = self.per_eps,
            period   = self.period,
        )                                     # (B, T, d)
        alpha_all = alpha_all.permute(1, 0, 2)  # → (T, B, d)

        S_all = self._build_S(alpha_all, K, V, d, B, T, device, dtype)
        return self._output(S_all, Q, d, B, T, dtype)

print("✅ VLATriton defined")

✅ VLATriton defined


---
## Block 6 — Benchmark

Measures wall-clock inference time for all three variants across sequence lengths T = 64 → 4096.

Expected results on Colab T4:
```
     T   Sequential     Parallel     Triton   Tri/Seq
    64       ~3ms          ~3ms       ~1ms      ~3x
   256      ~12ms          ~7ms       ~3ms      ~4x
  1024      ~48ms         ~20ms       ~9ms      ~5x
  4096     ~190ms         ~80ms      ~40ms      ~5x
```

In [10]:
def bench(layer, x, n=20):
    layer.eval()
    with torch.no_grad():
        for _ in range(5):
            layer(x)
    if x.device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n):
            layer(x)
    if x.device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000   # ms


d, B = 64, 4  # d must be power-of-2 ≤ 128 for Triton

print(f"d={d}  B={B}  device={DEVICE}\n")
print(f"{'T':>6}  {'Sequential':>12}  {'Parallel':>10}  {'Triton':>8}  {'Tri/Seq':>8}")
print("-" * 56)

for T in [64, 256, 512, 1024, 2048, 4096]:
    x   = torch.randn(B, T, d).to(DEVICE)
    seq = VLASequential(d_model=d).to(DEVICE)
    par = VLAParallel(d_model=d).to(DEVICE)

    t_seq = bench(seq, x)
    t_par = bench(par, x)

    if HAS_TRITON and DEVICE == "cuda":
        tri   = VLATriton(d_model=d).to(DEVICE)
        t_tri = bench(tri, x)
        print(f"{T:>6}  {t_seq:>10.1f}ms  {t_par:>8.1f}ms  {t_tri:>6.1f}ms  {t_seq/t_tri:>7.2f}x")
    else:
        print(f"{T:>6}  {t_seq:>10.1f}ms  {t_par:>8.1f}ms  {'N/A':>8}  {'N/A':>8}")

if DEVICE == "cpu":
    print("\nNOTE: parallel scan is SLOWER on CPU (tree overhead dominates at small d).")
    print("Run on GPU (Colab T4/A100) for real speedup numbers.")

d=64  B=4  device=cuda

     T    Sequential    Parallel    Triton   Tri/Seq
--------------------------------------------------------
    64        37.8ms      20.6ms     4.1ms     9.21x
   256       146.4ms      72.1ms     9.0ms    16.32x
   512       337.4ms     151.5ms    15.6ms    21.57x
  1024       621.4ms     269.6ms    27.9ms    22.31x
  2048      1221.6ms     578.6ms    54.4ms    22.45x
  4096      2465.9ms    1159.9ms   111.3ms    22.15x


---
## Block 7 — Gradient Check

Verifies gradients flow through all three paths cleanly — no NaN, no zero gradients.

In [11]:
print("=" * 60)
print("GRADIENT CHECK")
print("=" * 60)

d_grad = 64
x_grad = torch.randn(2, 64, d_grad).to(DEVICE)

for name, LayerClass, skip_cpu in [
    ("Sequential", VLASequential, False),
    ("Parallel",   VLAParallel,   False),
    ("Triton",     VLATriton,     True),
]:
    if skip_cpu and (not HAS_TRITON or DEVICE != "cuda"):
        print(f"\n[{name}]  skipped (needs CUDA + Triton)")
        continue

    layer = LayerClass(d_model=d_grad).to(DEVICE)
    out   = layer(x_grad)
    out.mean().backward()

    print(f"\n[{name}]")
    all_ok = True
    for pname, p in layer.named_parameters():
        if p.grad is not None:
            ok = torch.isfinite(p.grad).all().item()
            status = "✅" if ok else "❌ NaN!"
            if not ok:
                all_ok = False
            print(f"  {pname:40s}  norm={p.grad.norm():.4f}  {status}")
    if all_ok:
        print("  → All gradients finite ✅")

print("\nDone.")

GRADIENT CHECK

[Sequential]
  W_q.weight                                norm=0.1358  ✅
  W_q.bias                                  norm=0.0628  ✅
  W_k.weight                                norm=1.1167  ✅
  W_k.bias                                  norm=0.0940  ✅
  W_v.weight                                norm=0.1105  ✅
  W_v.bias                                  norm=0.0119  ✅
  out_norm.weight                           norm=0.0086  ✅
  out_norm.bias                             norm=0.0733  ✅
  W_o.weight                                norm=0.1091  ✅
  W_o.bias                                  norm=0.1250  ✅
  penalty.u_proj.weight                     norm=0.0340  ✅
  → All gradients finite ✅

[Parallel]
  W_q.weight                                norm=0.0819  ✅
  W_q.bias                                  norm=0.0662  ✅
  W_k.weight                                norm=0.1288  ✅
  W_k.bias                                  norm=0.0183  ✅
  W_v.weight                                nor

---
## Summary

| What | How | Why it matters |
|---|---|---|
| Sherman-Morrison | Rank-1 matrix inverse update, O(d²) per step | Exact, not approximate |
| Parallel scan | Blelloch binary tree, O(log T) depth | 2× GPU speedup |
| Triton kernel | Fuses T-step A_t loop into 1 kernel | Eliminates 4096 kernel launches |
| Unit-vector norm | F_t = I − k̂ α̂ᵀ | Prevents NaN in deep tree compositions |

### Stack for production inference
```
Python / PyTorch training
    └─ torch.onnx.export (dynamo=False, fixed T)
           └─ vla.onnx
                  └─ Rust (ort crate) + axum HTTP server
```

### Business angle in one line
> VLA is a drop-in O(N) replacement for attention that *learns what to forget* — making long-context LLMs trainable on consumer hardware and deployable without KV-cache infrastructure.

---
*Generated from: [variational-linear-attention](https://github.com/Vishal-sys-code/variational-linear-attention?tab=readme-ov-file)*